# L11 — Single-Server Queue: From Analysis to Simulation

**Module**: M05 | **Chapter**: 7 | **Lecture**: L11

## Learning Objectives
By the end of this notebook you will be able to:
1. Derive the M/M/1 steady-state distribution from balance equations.
2. Implement the M/M/1 queue from scratch in SimPy.
3. Verify simulated W, Wq, L, Lq, ρ against closed-form formulas.
4. Interpret the hockey-stick shape of Wq vs ρ.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 1. M/M/1 Analytical Formulas

For Poisson(λ) arrivals and Exponential(μ) service, one server:

| Quantity | Formula |
|---|---|
| Traffic intensity | ρ = λ/μ |
| Stability | ρ < 1 |
| P(n in system) | p_n = (1−ρ)ρⁿ |
| Mean in system | L = ρ/(1−ρ) |
| Mean in queue | Lq = ρ²/(1−ρ) |
| Mean time in system | W = 1/(μ−λ) |
| Mean wait in queue | Wq = λ/[μ(μ−λ)] |

In [ ]:
def mm1_theory(lam: float, mu: float) -> dict:
    """Compute M/M/1 analytical performance measures."""
    rho = lam / mu
    assert rho < 1.0, f"Unstable: ρ={rho:.3f} >= 1"
    return {
        'rho': rho,
        'L':   rho / (1 - rho),
        'Lq':  rho**2 / (1 - rho),
        'W':   1.0 / (mu - lam),
        'Wq':  lam / (mu * (mu - lam)),
    }

theory = mm1_theory(lam=3.0, mu=4.0)
print("M/M/1 theory (λ=3, μ=4):")
for k, v in theory.items():
    print(f"  {k:4s} = {v:.4f}")

## 2. SimPy Implementation

In [ ]:
def mm1_simulation(lam: float, mu: float, sim_time: float,
                   seed: int = 0) -> dict:
    """Simulate M/M/1 queue; return summary statistics dict."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    records = []
    busy_time = [0.0]

    def customer():
        arrival = env.now
        with server.request() as req:
            yield req
            wait = env.now - arrival
            svc = rng.exponential(1.0 / mu)
            yield env.timeout(svc)
            busy_time[0] += svc
        records.append({'wait': wait, 'sojourn': wait + svc})

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(customer())

    env.process(arrivals())
    env.run(until=sim_time)

    df = pd.DataFrame(records)
    Wq = df['wait'].mean()
    W  = df['sojourn'].mean()
    return {
        'rho': busy_time[0] / sim_time,
        'Wq': Wq,
        'W':  W,
        'Lq': lam * Wq,
        'L':  lam * W,
        'n_served': len(df),
    }

sim = mm1_simulation(lam=3.0, mu=4.0, sim_time=50_000, seed=42)
print("Simulation vs Theory (λ=3, μ=4, T=50,000):")
print(f"{'Metric':6s}  {'Sim':>8s}  {'Theory':>8s}  {'Rel err':>8s}")
print('-' * 40)
for k in ['rho', 'Wq', 'W', 'Lq', 'L']:
    t_val = theory[k]
    s_val = sim[k]
    err = abs(s_val - t_val) / t_val * 100
    print(f"{k:6s}  {s_val:8.4f}  {t_val:8.4f}  {err:7.2f}%")

## 3. The Hockey-Stick Curve

In [ ]:
# Sweep ρ from 0.1 to 0.95 and compare sim vs theory
mu = 5.0
rho_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sim_wq, theory_wq = [], []

for rho in rho_values:
    lam = rho * mu
    t = mm1_theory(lam, mu)
    s = mm1_simulation(lam, mu, sim_time=100_000, seed=0)
    theory_wq.append(t['Wq'] * mu)   # normalise by mu
    sim_wq.append(s['Wq'] * mu)

# Smooth theoretical curve
rho_fine = np.linspace(0.01, 0.97, 300)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rho_fine, rho_fine / (1 - rho_fine), 'b-', lw=2,
        label='M/M/1 theory: ρ/(1−ρ)')
ax.scatter(rho_values, sim_wq, color='red', zorder=5,
           label='Simulated (T=100,000)')
ax.set_xlabel('Utilisation ρ = λ/μ')
ax.set_ylabel('Normalised wait μWq')
ax.set_title('M/M/1 Hockey-Stick Curve')
ax.set_xlim(0, 1)
ax.set_ylim(0, 12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Little's Law Verification

In [ ]:
# Check L = lambda * W and Lq = lambda * Wq for multiple rho values
mu = 4.0
print(f"{'ρ':>5s}  {'λW':>8s}  {'L_sim':>8s}  {'λWq':>8s}  {'Lq_sim':>8s}")
print('-' * 50)
for rho in [0.3, 0.5, 0.7, 0.85]:
    lam = rho * mu
    s = mm1_simulation(lam, mu, sim_time=100_000, seed=1)
    print(f"{rho:>5.2f}  "
          f"{lam * s['W']:>8.4f}  {s['L']:>8.4f}  "
          f"{lam * s['Wq']:>8.4f}  {s['Lq']:>8.4f}")

---
## Try It Yourself

1. Run `mm1_simulation` with `lam=3.95, mu=4.0` (ρ=0.9875). The wait times will be very long and noisy. How large does `sim_time` need to be to estimate Wq within ±10% of theory with 95% confidence?

2. Implement a two-server version by changing `capacity=2`. Run with `lam=6.0, mu=4.0` (each server). Compare Wq to the M/M/2 Erlang-C formula (use the `erlang_c_wait_queue` method from `simdes.models.queues.MMCQueue`).

3. The M/G/1 Pollaczek-Khinchine formula says Wq = λE[S²] / (2(1−ρ)). Replace exponential service with lognormal(mean=1/mu, σ=1.0). Compute the theoretical Wq from P-K and verify with your simulation.